# 5.5.1 One Sample Hotelling's T² Statistic

# 1. Import necessary libraries

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import f, ttest_1samp
from numpy.linalg import inv

# 2. Create and load dataset

In [3]:
data = {
    'plant_var': ['A']*10 + ['B']*10 + ['C']*10 + ['D']*10,
    'height': [
        17.997, 17.362, 18.148, 19.023, 17.266, 17.266, 19.079, 18.267, 17.031, 18.043,  # Group A
        17.037, 17.034, 17.742, 15.587, 15.775, 16.938, 16.487, 17.814, 16.592, 16.088,  # Group B
        18.966, 17.274, 17.568, 16.075, 16.956, 17.611, 16.349, 17.876, 16.899, 17.208,  # Group C
        16.898, 19.352, 17.487, 16.442, 18.323, 16.279, 17.709, 15.540, 16.172, 17.697   # Group D
    ],
    'canopy_vol': [
        0.71, 0.81, 0.92, 0.61, 0.73, 0.77, 0.85, 0.65, 0.98, 0.82,  # Group A
        0.41, 0.56, 0.59, 0.68, 0.69, 0.67, 0.60, 0.62, 0.53, 0.61,  # Group B
        0.11, 0.51, 0.13, 0.23, 0.35, 0.21, 0.15, 0.48, 0.30, 0.33,  # Group C
        0.31, 0.51, 0.56, 0.59, 0.42, 0.35, 0.50, 0.52, 0.55, 0.54   # Group D
    ],
    'leaf_area': [
        15.0, 16.5, 15.2, 14.8, 16.0, 15.7, 16.3, 15.8, 16.1, 15.6,  # Group A
        14.5, 14.7, 15.2, 14.3, 14.6, 14.9, 14.8, 15.3, 14.6, 14.9,  # Group B
        15.4, 15.1, 15.0, 14.8, 15.2, 15.3, 14.9, 15.1, 14.7, 15.0,  # Group C
        15.6, 15.8, 15.7, 15.9, 16.0, 15.6, 15.8, 15.7, 15.9, 16.1   # Group D
    ]
}
df = pd.DataFrame(data) #Create DataFrame

# 3. Define function for one-sample Hotelling's T² test

In [4]:
# One-sample Hotelling's T² test function
def hotellings_t2_one_sample(X, mu):
    n, p = X.shape
    x_bar = np.mean(X, axis=0)
    S = np.cov(X, rowvar=False)
    T2 = n * np.dot(np.dot((x_bar - mu).T, inv(S)), (x_bar - mu))
    F_stat = (n - 1) * T2 / (n - p) / p
    p_value = 1 - f.cdf(F_stat, p, n - p)
    return T2, p_value

# 4. Define function for Post-hoc univariate t-tests (one-sample)

In [5]:
# Post-hoc univariate t-tests (one-sample t-tests)
def post_hoc_comparisons(X, mu, variable_names):
    post_hoc_results = []
    for i, var in enumerate(variable_names):
        t_stat, p_val = ttest_1samp(X[:, i], mu[i])
        post_hoc_results.append({'Variable': var, 't-statistic': t_stat, 'p-value': p_val})
    return pd.DataFrame(post_hoc_results)

# 5. Define function for Bonferroni correction

In [6]:
# Bonferroni correction function
def bonferroni_correction(p_values, num_tests):
    corrected_p = np.minimum(p_values * num_tests, 1.0)  # Ensure p-values don't exceed 1
    return corrected_p

# Hypothetical population mean (for one-sample test)
mu = np.array([17, 0.7, 15.5])

# Extract data for Group A
X_A = df[df['plant_var'] == 'A'][['height', 'canopy_vol', 'leaf_area']].values

# Perform one-sample Hotelling's T² test
T2_one_sample_A, p_one_sample_A = hotellings_t2_one_sample(X_A, mu)

print(f"Hotelling's T² Statistic: {T2_one_sample_A}")
print(f"p-value: {p_one_sample_A}")

Hotelling's T² Statistic: 37.52814050429714
p-value: 0.0016000332473309342


# 6. Post-hoc comparision

In [7]:
# If the Hotelling's T² test is significant then post-hoc comparisons
alpha = 0.05
if p_one_sample_A < alpha:
    print("\nHotelling's T² test is significant, performing post-hoc comparisons...\n")

    # post-hoc univariate t-tests
    variable_names = ['height', 'canopy_vol', 'leaf_area']
    post_hoc_results = post_hoc_comparisons(X_A, mu, variable_names)

    #Bonferroni correction
    num_tests = len(variable_names)  # Number of variables we are testing
    post_hoc_results['Bonferroni corrected p-value'] = bonferroni_correction(post_hoc_results['p-value'], num_tests)

    # post-hoc comparison
    print(post_hoc_results)
else:
    print("Hotelling's T² test is not significant. No post-hoc tests required.")


Hotelling's T² test is significant, performing post-hoc comparisons...

     Variable  t-statistic   p-value  Bonferroni corrected p-value
0      height     4.148170  0.002491                      0.007473
1  canopy_vol     2.327336  0.044943                      0.134828
2   leaf_area     1.129865  0.287745                      0.863235


# 5.5.2 Two Independent Sample Hotelling's T² Statistic

# 1. Import necessary libraries

In [8]:
import numpy as np
import pandas as pd
from scipy.stats import f, ttest_ind
from numpy.linalg import inv

# 2. Create and load dataset

In [9]:
data = {
    'plant_var': ['A']*10 + ['B']*10 + ['C']*10 + ['D']*10,
    'height': [
        17.997, 17.362, 18.148, 19.023, 17.266, 17.266, 19.079, 18.267, 17.031, 18.043,  # Group A
        17.037, 17.034, 17.742, 15.587, 15.775, 16.938, 16.487, 17.814, 16.592, 16.088,  # Group B
        18.966, 17.274, 17.568, 16.075, 16.956, 17.611, 16.349, 17.876, 16.899, 17.208,  # Group C
        16.898, 19.352, 17.487, 16.442, 18.323, 16.279, 17.709, 15.540, 16.172, 17.697   # Group D
    ],
    'canopy_vol': [
        0.71, 0.81, 0.92, 0.61, 0.73, 0.77, 0.85, 0.65, 0.98, 0.82,  # Group A
        0.41, 0.56, 0.59, 0.68, 0.69, 0.67, 0.60, 0.62, 0.53, 0.61,  # Group B
        0.11, 0.51, 0.13, 0.23, 0.35, 0.21, 0.15, 0.48, 0.30, 0.33,  # Group C
        0.31, 0.51, 0.56, 0.59, 0.42, 0.35, 0.50, 0.52, 0.55, 0.54   # Group D
    ],
    'leaf_area': [
        15.0, 16.5, 15.2, 14.8, 16.0, 15.7, 16.3, 15.8, 16.1, 15.6,  # Group A
        14.5, 14.7, 15.2, 14.3, 14.6, 14.9, 14.8, 15.3, 14.6, 14.9,  # Group B
        15.4, 15.1, 15.0, 14.8, 15.2, 15.3, 14.9, 15.1, 14.7, 15.0,  # Group C
        15.6, 15.8, 15.7, 15.9, 16.0, 15.6, 15.8, 15.7, 15.9, 16.1   # Group D
    ]
}
df = pd.DataFrame(data)

# 3. Define function for two-sample Hotelling's T² test

In [10]:
# Two-sample Hotelling's T² test
def hotellings_t2_two_sample(X, Y):
    n1, p = X.shape
    n2 = Y.shape[0]
    X_bar = np.mean(X, axis=0)
    Y_bar = np.mean(Y, axis=0)
    S1 = np.cov(X, rowvar=False)
    S2 = np.cov(Y, rowvar=False)

    # Pooled covariance matrix
    Sp = ((n1 - 1) * S1 + (n2 - 1) * S2) / (n1 + n2 - 2)

    # Hotelling's T² statistic
    diff = X_bar - Y_bar
    T2 = (n1 * n2) / (n1 + n2) * np.dot(np.dot(diff.T, inv(Sp)), diff)

    # Convert to F-statistic
    F_stat = (n1 + n2 - p - 1) * T2 / ((n1 + n2 - 2) * p)
    p_value = 1 - f.cdf(F_stat, p, n1 + n2 - p - 1)
    return T2, p_value

# 4. Define function for post-hoc univariate t-tests (two-sample)

In [11]:
# Post-hoc univariate t-tests (two-sample t-tests)
def post_hoc_comparisons(X, Y, variable_names):
    post_hoc_results = []
    for i, var in enumerate(variable_names):
        t_stat, p_val = ttest_ind(X[:, i], Y[:, i], equal_var=False)
        post_hoc_results.append({'Variable': var, 't-statistic': t_stat, 'p-value': p_val})
    return pd.DataFrame(post_hoc_results)

# 5. Define function for Bonferroni correction

In [12]:
# Bonferroni correction
def bonferroni_correction(p_values, num_tests):
    corrected_p = np.minimum(p_values * num_tests, 1.0)  # Ensure p-values don't exceed 1
    return corrected_p

# Extract data for Group A and Group B
X_A = df[df['plant_var'] == 'A'][['height', 'canopy_vol', 'leaf_area']].values
X_B = df[df['plant_var'] == 'B'][['height', 'canopy_vol', 'leaf_area']].values

# two-sample Hotelling's T² test
T2_two_sample, p_two_sample = hotellings_t2_two_sample(X_A, X_B)

print(f"Hotelling's T² Statistic: {T2_two_sample}")
print(f"p-value: {p_two_sample}")

Hotelling's T² Statistic: 56.01588069740404
p-value: 3.6156694564026814e-05


# 6. Post-hoc comparision

In [28]:
# Hotelling's T² test is significant then post-hoc
alpha = 0.05
if p_two_sample < alpha:
    print("\nHotelling's T² test is significant, performing post-hoc comparisons...\n")

    # post-hoc univariate t-tests
    variable_names = ['height', 'canopy_vol', 'leaf_area']
    post_hoc_results = post_hoc_comparisons(X_A, X_B, variable_names)

    # Bonferroni correction
    num_tests = len(variable_names)  # Number of variables we are testing
    post_hoc_results['Bonferroni corrected p-value'] = bonferroni_correction(post_hoc_results['p-value'], num_tests)

    # post-hoc comparison
    print(post_hoc_results)
else:
    print("Hotelling's T² test is not significant. No post-hoc tests required.")


Hotelling's T² test is significant, performing post-hoc comparisons...

     Variable  t-statistic   p-value  Bonferroni corrected p-value
0      height     3.747231  0.001479                      0.004437
1  canopy_vol     4.197237  0.000652                      0.001956
2   leaf_area     4.552200  0.000452                      0.001355


# 5.5.3  Paired-Sample Hotelling's T² Statistic

# 1. Import necessary libraries

In [29]:
import pandas as pd
from scipy import linalg
import numpy as np
from scipy.stats import f

# 2. Create and load the dataset

In [30]:
# paired dataset
data = {
    'plant_var': ['A']*10 + ['B']*10 + ['C']*10 + ['D']*10,
    'height_before': [
        17.997, 17.362, 18.148, 19.023, 17.266, 17.266, 19.079, 18.267, 17.031, 18.043,  # Group A
        17.037, 17.034, 17.742, 15.587, 15.775, 16.938, 16.487, 17.814, 16.592, 16.088,  # Group B
        18.966, 17.274, 17.568, 16.075, 16.956, 17.611, 16.349, 17.876, 16.899, 17.208,  # Group C
        16.898, 19.352, 17.487, 16.442, 18.323, 16.279, 17.709, 15.540, 16.172, 17.697   # Group D
    ],
    'height_after': [
        18.497, 17.862, 18.648, 19.523, 17.766, 17.766, 19.579, 18.767, 17.531, 18.543,  # Group A
        17.537, 17.534, 18.242, 16.087, 16.275, 17.438, 16.987, 18.314, 17.092, 16.588,  # Group B
        19.466, 17.774, 18.068, 16.575, 17.456, 18.111, 16.849, 18.376, 17.399, 17.708,  # Group C
        17.398, 19.852, 17.987, 16.942, 18.823, 16.779, 18.209, 16.040, 16.672, 18.197   # Group D
    ],
    'leaf_area_before': [
        15.0, 16.5, 15.2, 14.8, 16.0, 15.7, 16.3, 15.8, 16.1, 15.6,  # Group A
        14.5, 14.7, 15.2, 14.3, 14.6, 14.9, 14.8, 15.3, 14.6, 14.9,  # Group B
        15.4, 15.1, 15.0, 14.8, 15.2, 15.3, 14.9, 15.1, 14.7, 15.0,  # Group C
        15.6, 15.8, 15.7, 15.9, 16.0, 15.6, 15.8, 15.7, 15.9, 16.1   # Group D
    ],
    'leaf_area_after': [
        15.5, 16.8, 15.7, 15.2, 16.5, 16.2, 16.8, 16.3, 16.6, 16.1,  # Group A
        15.0, 15.2, 15.7, 14.8, 15.1, 15.4, 15.3, 15.8, 15.1, 15.4,  # Group B
        15.9, 15.6, 15.5, 15.3, 15.7, 15.8, 15.4, 15.6, 15.2, 15.5,  # Group C
        16.1, 16.3, 16.2, 16.4, 16.5, 16.1, 16.3, 16.2, 16.4, 16.6   # Group D
    ]
}

# DataFrame
df = pd.DataFrame(data)
print(df)


   plant_var  height_before  height_after  leaf_area_before  leaf_area_after
0          A         17.997        18.497              15.0             15.5
1          A         17.362        17.862              16.5             16.8
2          A         18.148        18.648              15.2             15.7
3          A         19.023        19.523              14.8             15.2
4          A         17.266        17.766              16.0             16.5
5          A         17.266        17.766              15.7             16.2
6          A         19.079        19.579              16.3             16.8
7          A         18.267        18.767              15.8             16.3
8          A         17.031        17.531              16.1             16.6
9          A         18.043        18.543              15.6             16.1
10         B         17.037        17.537              14.5             15.0
11         B         17.034        17.534              14.7             15.2

# 3. Define function for paired Hotelling's T² test

In [31]:
# Extract 'Before' and 'After' data for height and leaf_area
before_data = df[['height_before', 'leaf_area_before']].values
after_data = df[['height_after', 'leaf_area_after']].values

# Calculate the differences between After and Before
diff_data = after_data - before_data

# Number of samples (plants)
n = diff_data.shape[0]

# Mean difference vector
mean_diff = np.mean(diff_data, axis=0)

# Covariance matrix of the differences
cov_diff = np.cov(diff_data, rowvar=False)

# Hotelling's T-squared statistic
T_squared = n * mean_diff.T @ linalg.inv(cov_diff) @ mean_diff

# F-statistic approximation
p = 2  # number of variables (height and leaf_area)
F_stat = (n - p) / (p * (n - 1)) * T_squared

# P-value
p_value = 1 - f.cdf(F_stat, p, n - p)

print(f"Hotelling's T-squared statistic: {T_squared}")
print(f"F-statistic: {F_stat}")
print(f"p-value: {p_value}")

# Interpretation
if p_value < 0.05:
    print("Significant difference between the two conditions (Before and After) for height and leaf_area.")
else:
    print("No significant difference between the two conditions (Before and After) for height and leaf_area.")

Hotelling's T-squared statistic: 1.237417025435734e+32
F-statistic: 6.028441918789473e+31
P-value: 0.0
Significant difference between the two conditions (Before and After) for height and leaf_area.
